## Theta trigger latency check (recorded replay vs. real `theta_trigger_loop`)

This mirrors `tests/theta_lifu_validation_test.py::test_recorded_trigger_replay_matches_online_markers`
exactly, by importing and reusing that test module's helpers directly, rather than re-implementing
the buffer/median/MAD/threshold trigger logic by hand again (the old version of this notebook did
that, which is why it used to open with a comment saying it "isn't the same as theta_trigger_loop" --
it wasn't actually running the same code, just an approximation of it). Instead of asserting
pass/fail like the test does, this prints the online (actually recorded) vs. offline (replayed
through the real `theta_trigger_loop`) LIFU_ON time and the resulting latency for every sonication,
across every recorded run.


In [117]:
import sys
import logging
from pathlib import Path

import numpy as np

REPO_ROOT = Path(r"C:\Users\jshin\OW_closedloopLIFU")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests import theta_lifu_validation_test as ttest

PARTICIPANTS = ["dave", "scott"]
RUNS = [1, 2, 3, 4]
XDF_ROOT = REPO_ROOT / "xdf_data"


def xdf_path_for(participant, run):
    folder = f"sub-{participant}_run_{run}"
    fname = f"sub-{participant}_run_{run}_ses-1_task-{participant}_run_{run}_run-001_eeg.xdf"
    return XDF_ROOT / folder / "ses-1" / "eeg" / fname


In [118]:
def run_recording(xdf_path):
    """Replays one recording through the real theta_trigger_loop -- exactly like
    ttest.test_recorded_trigger_replay_matches_online_markers -- and returns
    (online_t, offline_t, latency_ms) for every matched LIFU_ON pair, times in
    seconds relative to the recording start.
    """
    ttest.XDF_PATH = xdf_path
    streams = ttest._load_streams()
    eeg_stream = streams[ttest.EEG_STREAM_NAME]
    lifu_stream = streams[ttest.TRIGGER_STREAM_NAME]
    t0 = eeg_stream["time_stamps"][0]

    sonication_start_ts = ttest._find_marker_time(lifu_stream, "START_EXPERIMENT_RECEIVED")
    if sonication_start_ts is None:
        sonication_start_ts = t0

    ttest.main_pipeline.psychopy_running = False
    current_ts_holder = {"ts": None}
    sample_source = ttest._recorded_sample_source(eeg_stream, current_ts_holder, sonication_start_ts)

    # theta_trigger_loop logs an INFO line for every rejected sample (there can be
    # thousands per recording) -- pytest quietly swallows those by default, but a
    # notebook kernel actually has to serialize and display each one, which is both
    # the noise and most of the slowness. Raising the level to WARNING for the
    # duration of the replay drops them without touching main_pipeline itself.
    previous_level = ttest.main_pipeline.logger.level
    ttest.main_pipeline.logger.setLevel(logging.WARNING)
    try:
        events = ttest._run_with_sample_source(sample_source, current_ts_holder)
    finally:
        ttest.main_pipeline.logger.setLevel(previous_level)

    offline_on = sorted(ts - t0 for marker, ts in events if marker == "LIFU_ON")
    online_on = sorted(
        ts - t0
        for marker, ts in zip((m[0] for m in lifu_stream["time_series"]), lifu_stream["time_stamps"])
        if marker == "LIFU_ON"
    )

    if len(offline_on) != len(online_on):
        print(f"  WARNING: {len(offline_on)} offline vs {len(online_on)} online LIFU_ON -- count mismatch.")

    n = min(len(offline_on), len(online_on))
    return [(online_on[i], offline_on[i], (online_on[i] - offline_on[i]) * 1000) for i in range(n)]


In [119]:
all_rows = []
for participant in PARTICIPANTS:
    for run in RUNS:
        xdf_path = xdf_path_for(participant, run)
        if not xdf_path.exists():
            print(f"=== {participant} run {run}: fixture not found, skipping ===")
            continue
        print(f"\n=== {participant} run {run}: {xdf_path.name} ===")
        for online_t, offline_t, latency_ms in run_recording(xdf_path):
            all_rows.append({
                "participant": participant, "run": run,
                "online_t": online_t, "offline_t": offline_t, "latency_ms": latency_ms,
            })

print("\n\n=== All sonications, every run ===")
header = f"{'participant':<10}{'run':<5}{'online (s)':>12}{'offline (s)':>12}{'latency (ms)':>14}"
print(header)
for row in all_rows:
    print(f"{row['participant']:<10}{row['run']:<5}{row['online_t']:>12.3f}"
          f"{row['offline_t']:>12.3f}{row['latency_ms']:>14.2f}")

if all_rows:
    latencies = np.array([row["latency_ms"] for row in all_rows])
    print(f"\nLatency summary over {len(latencies)} sonication(s): "
          f"mean={latencies.mean():.2f}ms  median={np.median(latencies):.2f}ms  "
          f"min={latencies.min():.2f}ms  max={latencies.max():.2f}ms  std={latencies.std():.2f}ms")
else:
    print("No matched LIFU_ON events found in any run.")



=== dave run 1: sub-dave_run_1_ses-1_task-dave_run_1_run-001_eeg.xdf ===



=== dave run 2: sub-dave_run_2_ses-1_task-dave_run_2_run-001_eeg.xdf ===



=== dave run 3: sub-dave_run_3_ses-1_task-dave_run_3_run-001_eeg.xdf ===



=== dave run 4: sub-dave_run_4_ses-1_task-dave_run_4_run-001_eeg.xdf ===



=== scott run 1: sub-scott_run_1_ses-1_task-scott_run_1_run-001_eeg.xdf ===



=== scott run 2: sub-scott_run_2_ses-1_task-scott_run_2_run-001_eeg.xdf ===



=== scott run 3: sub-scott_run_3_ses-1_task-scott_run_3_run-001_eeg.xdf ===



=== scott run 4: sub-scott_run_4_ses-1_task-scott_run_4_run-001_eeg.xdf ===




=== All sonications, every run ===
participantrun    online (s) offline (s)  latency (ms)
dave      1          96.384      96.382          1.35
dave      1         106.541     106.540          0.66
dave      1         116.670     116.668          1.47
dave      1         131.583     131.582          1.54
dave      1         146.344     146.343          0.81
dave      1         156.426     156.423          3.69
dave      2          55.717      55.717          0.44
dave      2          68.479      68.479          0.70
dave      2         153.008     153.007          0.85
dave      2         180.928     180.927          0.57
dave      2         196.125     196.125          0.53
dave      4          52.675      52.674          1.42
dave      4          67.834      63.268       4566.39
dave      4          84.077      73.599      10478.07
dave      4          99.388      84.075      15313.35
dave      4         116.075      94.749      21325.84
dave      4         181.514     105.229     